# Navigation Styles Assignment for Trajectories

This notebook assigns navigation styles to trajectories based on their movement characteristics and POI interactions.

## Navigation Style Definitions

The following navigation styles are assigned based on specific criteria:

- **Casual**: Trajectories with an average speed below 3.5 (units from data)
- **Exploratory**: Trajectories with more than 10 meaningful interactions
- **Immersive**: Trajectories where POI interaction durations exceed the typical duration for that POI category
- **Targeted**: Trajectories with exactly 2 meaningful interactions

A trajectory can have multiple navigation styles if it meets multiple criteria.

## Section 1: Imports and Fuseki Connection Setup

Import required libraries and configure the connection to the Apache Fuseki endpoint.

In [ ]:
# Standard library imports
import logging
from collections import defaultdict
from pathlib import Path

# Third-party imports
import pandas as pd
from helpers.fuseki_config import query_endpoint, update_endpoint

# Project imports - helper modules
from helpers.fuseki_queries import execute_sparql_query, execute_sparql_update, format_results

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Display configuration
print(f"Fuseki Query Endpoint: {query_endpoint}")
print(f"Fuseki Update Endpoint: {update_endpoint}")
print("\nNavigation Styles Thresholds:")
print("- Casual: Average speed < 3.5")
print("- Exploratory: More than 10 meaningful interactions")
print("- Immersive: Interaction duration > Typical duration for POI category")
print("- Targeted: Exactly 2 meaningful interactions")

## Section 2: SPARQL Queries to Fetch Trajectory Data

Define and execute SPARQL queries to retrieve:
1. Trajectory average speeds
2. Meaningful interactions count per trajectory
3. POI interaction details with durations and typical durations

In [ ]:
# Query 1: Get average speed for each trajectory
avg_speed_query = """
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX datacron: <http://www.datacron-project.eu/datAcron#>
PREFIX dul: <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?trajectory (AVG(xsd:float(?speed)) AS ?avgSpeed)
WHERE {
    ?trajectory a datacron:Trajectory .
    ?position dul:isPartOf ?trajectory .
    ?position datacron:hasSpeed ?speed .
}
GROUP BY ?trajectory
"""

logger.info("Executing query to fetch trajectory average speeds...")
try:
    avg_speed_results = execute_sparql_query(query_endpoint, avg_speed_query)
    avg_speed_df = format_results(avg_speed_results)

    if not avg_speed_df.empty:
        avg_speed_df['avgSpeed'] = pd.to_numeric(avg_speed_df['avgSpeed'], errors='coerce')
        logger.info(f"Retrieved average speeds for {len(avg_speed_df)} trajectories")
        print("\nSample trajectory speeds:")
        print(avg_speed_df.head())
        print("\nSpeed statistics:")
        print(avg_speed_df['avgSpeed'].describe())
    else:
        logger.warning("No trajectory speed data found")
        avg_speed_df = pd.DataFrame(columns=['trajectory', 'avgSpeed'])

except Exception as e:
    logger.error(f"Error fetching trajectory speeds: {e}")
    avg_speed_df = pd.DataFrame(columns=['trajectory', 'avgSpeed'])

In [ ]:
# Query 2: Count meaningful interactions per trajectory
interaction_count_query = """
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX poi: <http://w3id.org/trajecto/poi#>
PREFIX datacron: <http://www.datacron-project.eu/datAcron#>
PREFIX dul: <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>

SELECT ?trajectory (COUNT(DISTINCT ?interaction) AS ?interactionCount)
WHERE {
    ?interaction a poi:MeaningfulInteraction .
    ?position trajecto:hasInteraction ?interaction ;
              dul:isPartOf ?trajectory .
}
GROUP BY ?trajectory
"""

logger.info("Executing query to count meaningful interactions per trajectory...")
try:
    interaction_count_results = execute_sparql_query(query_endpoint, interaction_count_query)
    interaction_count_df = format_results(interaction_count_results)

    if not interaction_count_df.empty:
        interaction_count_df['interactionCount'] = pd.to_numeric(
            interaction_count_df['interactionCount'],
            errors='coerce'
        )
        logger.info(f"Retrieved interaction counts for {len(interaction_count_df)} trajectories")
        print("\nSample interaction counts:")
        print(interaction_count_df.head())
        print("\nInteraction count statistics:")
        print(interaction_count_df['interactionCount'].describe())
    else:
        logger.warning("No interaction count data found")
        interaction_count_df = pd.DataFrame(columns=['trajectory', 'interactionCount'])

except Exception as e:
    logger.error(f"Error fetching interaction counts: {e}")
    interaction_count_df = pd.DataFrame(columns=['trajectory', 'interactionCount'])

In [ ]:
# ============================================================================
# STEP: Classify Immersive Interactions in Triplestore
# ============================================================================
# Before querying for immersive trajectories, we need to classify which
# interactions are immersive based on dwell time vs. typical duration.
# This is done by executing the immersive-interaction.sparql update query.

# Read the SPARQL update query from file
immersive_classification_path = Path('../sparql/immersive-interaction.sparql')


with open(immersive_classification_path) as f:
    immersive_classification_query = f.read()

logger.info(f"Loaded SPARQL query from: {immersive_classification_path}")
print("\nExecuting immersive interaction classification...")

# Execute the update query to classify immersive interactions
classification_success = execute_sparql_update(
    update_endpoint,
    immersive_classification_query
)

if classification_success:
    logger.info("Successfully classified immersive interactions in triplestore")
    print("Status: ✓ Immersive interactions classified successfully")

    # Verify how many immersive interactions were created
    count_query = """
    PREFIX poi: <http://w3id.org/trajecto/poi#>

    SELECT (COUNT(?interaction) AS ?immersiveCount)
    WHERE {
        ?interaction a poi:ImmersiveInteraction .
    }
    """

    count_results = execute_sparql_query(query_endpoint, count_query)
    count_df = format_results(count_results)

    if not count_df.empty and 'immersiveCount' in count_df.columns:
        immersive_count = int(count_df['immersiveCount'].iloc[0])
        print(f"Total immersive interactions identified: {immersive_count}")
    else:
        print("Could not retrieve count of immersive interactions")
else:
    logger.error("Failed to classify immersive interactions")
    print("Status: ✗ Classification failed")

In [ ]:
# Query 3: Get trajectories with immersive interactions
# This query retrieves trajectories that have at least one interaction
# classified as poi:ImmersiveInteraction (from the previous classification step)

immersive_trajectories_query = """
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX poi: <http://w3id.org/trajecto/poi#>
PREFIX datacron: <http://www.datacron-project.eu/datAcron#>
PREFIX dul: <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>

SELECT DISTINCT ?trajectory
WHERE {
    ?interaction a poi:ImmersiveInteraction .
    ?position trajecto:hasInteraction ?interaction ;
              dul:isPartOf ?trajectory .
}
"""

logger.info("Executing query to fetch trajectories with immersive interactions...")
try:
    immersive_traj_results = execute_sparql_query(query_endpoint, immersive_trajectories_query)
    immersive_traj_df = format_results(immersive_traj_results)

    if not immersive_traj_df.empty:
        logger.info(f"Retrieved {len(immersive_traj_df)} trajectories with immersive interactions")
        print("\nTrajectories with Immersive Interactions:")
        print(f"Count: {len(immersive_traj_df)}")
        print("\nSample trajectories:")
        print(immersive_traj_df.head())
    else:
        logger.warning("No trajectories with immersive interactions found")
        immersive_traj_df = pd.DataFrame(columns=['trajectory'])
        print("\nNo immersive trajectories identified")

except Exception as e:
    logger.error(f"Error fetching immersive trajectories: {e}")
    immersive_traj_df = pd.DataFrame(columns=['trajectory'])

## Section 3: Data Processing and Navigation Style Classification

Process the retrieved data and classify each trajectory according to the navigation style rules.

In [ ]:
# Classification 1: Casual Navigation Style (Average speed < 3.5)
CASUAL_SPEED_THRESHOLD = 3.5

casual_trajectories = set()

if not avg_speed_df.empty:
    casual_mask = avg_speed_df['avgSpeed'] < CASUAL_SPEED_THRESHOLD
    casual_trajectories = set(avg_speed_df[casual_mask]['trajectory'].tolist())

    logger.info(f"Identified {len(casual_trajectories)} Casual trajectories")
    print("\nCasual Navigation Style:")
    print(f"Threshold: Average speed < {CASUAL_SPEED_THRESHOLD}")
    print(f"Trajectories identified: {len(casual_trajectories)}")

    if len(casual_trajectories) > 0:
        casual_speeds = avg_speed_df[casual_mask]['avgSpeed']
        print(f"Speed range: {casual_speeds.min():.2f} - {casual_speeds.max():.2f}")
else:
    logger.warning("No speed data available for Casual classification")

In [ ]:
# Classification 2: Exploratory Navigation Style (More than 10 meaningful interactions)
EXPLORATORY_INTERACTION_THRESHOLD = 10

exploratory_trajectories = set()

if not interaction_count_df.empty:
    exploratory_mask = interaction_count_df['interactionCount'] > EXPLORATORY_INTERACTION_THRESHOLD
    exploratory_trajectories = set(interaction_count_df[exploratory_mask]['trajectory'].tolist())

    logger.info(f"Identified {len(exploratory_trajectories)} Exploratory trajectories")
    print("\nExploratory Navigation Style:")
    print(f"Threshold: More than {EXPLORATORY_INTERACTION_THRESHOLD} meaningful interactions")
    print(f"Trajectories identified: {len(exploratory_trajectories)}")

    if len(exploratory_trajectories) > 0:
        exploratory_counts = interaction_count_df[exploratory_mask]['interactionCount']
        print(f"Interaction count range: {int(exploratory_counts.min())} - {int(exploratory_counts.max())}")
else:
    logger.warning("No interaction count data available for Exploratory classification")

In [ ]:
# Classification 3: Targeted Navigation Style (1 or 2 meaningful interactions)
TARGETED_INTERACTION_COUNT = 2

targeted_trajectories = set()

if not interaction_count_df.empty:
    targeted_mask = interaction_count_df['interactionCount'] <= TARGETED_INTERACTION_COUNT
    targeted_trajectories = set(interaction_count_df[targeted_mask]['trajectory'].tolist())

    logger.info(f"Identified {len(targeted_trajectories)} Targeted trajectories")
    print("\nTargeted Navigation Style:")
    print(f"Threshold: {TARGETED_INTERACTION_COUNT} meaningful interactions")
    print(f"Trajectories identified: {len(targeted_trajectories)}")
else:
    logger.warning("No interaction count data available for Targeted classification")

In [ ]:
# Consolidate all classification results
trajectory_styles: dict[str, list[str]] = defaultdict(list)

for trajectory in casual_trajectories:
    trajectory_styles[trajectory].append('Casual')

for trajectory in exploratory_trajectories:
    trajectory_styles[trajectory].append('Exploratory')

for trajectory in targeted_trajectories:
    trajectory_styles[trajectory].append('Targeted')

# for trajectory in immersive_trajectories:
#     trajectory_styles[trajectory].append('Immersive')

# Create summary DataFrame
summary_data = []
for trajectory, styles in trajectory_styles.items():
    summary_data.append({
        'trajectory': trajectory,
        'navigation_styles': ', '.join(styles),
        'style_count': len(styles)
    })

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*80)
print("CLASSIFICATION SUMMARY")
print("="*80)
print(f"\nTotal trajectories classified: {len(trajectory_styles)}")
print("\nNavigation Style Distribution:")
print(f"- Casual: {len(casual_trajectories)} trajectories")
print(f"- Exploratory: {len(exploratory_trajectories)} trajectories")
print(f"- Targeted: {len(targeted_trajectories)} trajectories")
# print(f"- Immersive: {len(immersive_trajectories)} trajectories")

if not summary_df.empty:
    print("\nMultiple Styles per Trajectory:")
    style_count_dist = summary_df['style_count'].value_counts().sort_index()
    for count, freq in style_count_dist.items():
        print(f"- {count} style(s): {freq} trajectories")

    print("\nSample Classifications:")
    print(summary_df.head(10))
else:
    logger.warning("No trajectories were classified")

## Section 4: SPARQL UPDATE Queries to Assign Navigation Styles

Update the knowledge graph by assigning navigation style classifications to trajectories.

In [ ]:
def assign_navigation_style(trajectory_uri: str, style: str) -> bool:
    """
    Assign a navigation style to a trajectory in the knowledge graph.

    Args:
        trajectory_uri: URI of the trajectory
        style: Navigation style name (Casual, Exploratory, Targeted, or Immersive)

    Returns:
        bool: True if successful, False otherwise
    """
    # Map style names to ontology classes
    style_class_map = {
        'Casual': 'patterns:CasualMovement',
        'Exploratory': 'patterns:ExploratoryMovement',
        'Targeted': 'patterns:TargetedSegment',
        'Immersive': 'patterns:ImmersiveMovement'
    }

    if style not in style_class_map:
        logger.error(f"Unknown navigation style: {style}")
        return False

    style_class = style_class_map[style]

    update_query = f"""
    PREFIX trajecto: <http://w3id.org/trajecto#>
    PREFIX patterns: <http://w3id.org/trajecto/patterns#>
    PREFIX datacron: <http://www.datacron-project.eu/datAcron#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    INSERT DATA {{
        <{trajectory_uri}> patterns:exhibitsPattern {style_class} .
    }}
    """

    try:
        result = execute_sparql_update(update_endpoint, update_query)
        return result
    except Exception as e:
        logger.error(f"Error assigning {style} style to {trajectory_uri}: {e}")
        return False

In [ ]:
# Execute updates to assign navigation styles to trajectories
print("\n" + "="*80)
print("ASSIGNING NAVIGATION STYLES TO KNOWLEDGE GRAPH")
print("="*80)

success_count = 0
failure_count = 0
total_assignments = sum(len(styles) for styles in trajectory_styles.values())

logger.info(f"Starting to assign {total_assignments} navigation style assignments...")

for trajectory_uri, styles in trajectory_styles.items():
    trajectory_name = trajectory_uri.split('#')[-1]
    print(f"\nProcessing {trajectory_name}...")

    for style in styles:
        print(f"  - Assigning '{style}' style...", end=" ")

        if assign_navigation_style(trajectory_uri, style):
            print("✓")
            success_count += 1
        else:
            print("✗")
            failure_count += 1

print("\n" + "="*80)
print("ASSIGNMENT RESULTS")
print("="*80)
print(f"Total assignments attempted: {total_assignments}")
print(f"Successful: {success_count}")
print(f"Failed: {failure_count}")
print(f"Success rate: {(success_count/total_assignments*100):.1f}%" if total_assignments > 0 else "N/A")

if success_count > 0:
    logger.info(f"Successfully assigned {success_count} navigation styles to trajectories")
else:
    logger.warning("No navigation styles were successfully assigned")

## Section 5: Verify Navigation Style Assignments

Query the knowledge graph to verify that navigation styles were successfully assigned to trajectories.

In [ ]:
# Verify navigation style assignments in the knowledge graph
verification_query = """
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX patterns: <http://w3id.org/trajecto/patterns#>
PREFIX datacron: <http://www.datacron-project.eu/datAcron#>

SELECT ?trajectory ?pattern
WHERE {
    ?trajectory a datacron:Trajectory ;
                patterns:exhibitsPattern ?pattern .
}
ORDER BY ?trajectory
"""

logger.info("Verifying navigation style assignments in knowledge graph...")
try:
    verification_results = execute_sparql_query(query_endpoint, verification_query)
    verification_df = format_results(verification_results)

    if not verification_df.empty:
        print(f"\n✓ Found {len(verification_df)} navigation style assignments in knowledge graph")

        # Count assignments per style
        style_counts = verification_df['pattern'].value_counts()
        print("\nNavigation Style Assignments:")
        for pattern, count in style_counts.items():
            style_name = pattern.split('#')[-1] if '#' in pattern else pattern
            print(f"  - {style_name}: {count} trajectories")

        # Show sample assignments
        print("\nSample Assignments:")
        print(verification_df.head(10))

        # Count trajectories with multiple styles
        trajectory_counts = verification_df.groupby('trajectory').size()
        multi_style_count = (trajectory_counts > 1).sum()
        print(f"\nTrajectories with multiple styles: {multi_style_count}")

    else:
        print("\n✗ No navigation style assignments found in knowledge graph")
        logger.warning("Verification found no assignments")

except Exception as e:
    logger.error(f"Error during verification: {e}")
    print(f"✗ Verification failed: {e}")

In [ ]:
# Validate KG state against stage-06-navigation SHACL shapes
from helpers.shacl_validator import validate_before_import, construct_validation_graph

proposed = construct_validation_graph('stage-06-navigation')
print(f"Fetched {len(proposed)} triples for validation")
report = validate_before_import(proposed, 'stage-06-navigation', halt_on_violations=True)